# Step 3: YAMNet from Kaggle, Embedding Extraction, and Prototype Inference

This notebook uses the pretrained YAMNet model from Kaggle as the frozen embedding backbone. The notebook downloads the model with Kaggle tooling, extracts 1024-dimensional embeddings, builds a prototype store from training clips, evaluates nearest-prototype inference, and exports the resulting artifacts.

Before running the download cell, make sure Kaggle credentials are configured in the notebook environment.

In [ ]:
%pip install -q tensorflow kagglehub pandas numpy librosa soundfile scikit-learn

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import kagglehub
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
from sklearn.metrics import classification_report

PROJECT_ROOT = Path.cwd().resolve().parents[1]
ML_ROOT = PROJECT_ROOT / 'ml'
CONTRACTS_DIR = PROJECT_ROOT / 'packages' / 'shared_contracts' / 'v1'
CURATED_MANIFEST_PATH = ML_ROOT / 'data' / 'curated' / 'soundpulse_manifest.v1.json'
YAMNET_OUTPUT_DIR = ML_ROOT / 'export' / 'artifacts' / 'generated' / 'yamnet_kaggle'
YAMNET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with (CONTRACTS_DIR / 'class_labels.json').open('r', encoding='utf-8') as handle:
    CLASS_CONTRACT = json.load(handle)
with (CONTRACTS_DIR / 'preprocessing_config.json').open('r', encoding='utf-8') as handle:
    PREPROCESSING_CONTRACT = json.load(handle)

CLASS_ORDER = [item['slug'] for item in CLASS_CONTRACT['classes']]
CLASS_TO_INDEX = {slug: index for index, slug in enumerate(CLASS_ORDER)}
SAMPLE_RATE = PREPROCESSING_CONTRACT['audio']['sample_rate_hz']
WINDOW_DURATION_MS = PREPROCESSING_CONTRACT['audio']['window_duration_ms']
WINDOW_SAMPLES = int(SAMPLE_RATE * WINDOW_DURATION_MS / 1000)
YAMNET_KAGGLE_HANDLE = os.environ.get('SOUNDPULSE_YAMNET_KAGGLE_HANDLE', 'google/yamnet/tensorFlow2/yamnet')

assert CURATED_MANIFEST_PATH.exists(), 'Run Notebook 1 first to build the curated manifest.'
print({'yamnet_kaggle_handle': YAMNET_KAGGLE_HANDLE, 'sample_rate_hz': SAMPLE_RATE})

In [ ]:
if not os.environ.get('KAGGLE_USERNAME') or not os.environ.get('KAGGLE_KEY'):
    print('Kaggle credentials are not set in this environment. Set KAGGLE_USERNAME and KAGGLE_KEY before running the download cell.')

yamnet_download_dir = Path(kagglehub.model_download(YAMNET_KAGGLE_HANDLE))
yamnet_download_dir

In [ ]:
def find_yamnet_artifact(model_root: Path) -> Path:
    candidates = list(model_root.rglob('saved_model.pb')) + list(model_root.rglob('*.keras')) + list(model_root.rglob('*.h5'))
    if not candidates:
        raise FileNotFoundError(f'No loadable YAMNet artifact found under {model_root}')
    return candidates[0]


yamnet_artifact = find_yamnet_artifact(yamnet_download_dir)
yamnet_artifact

In [ ]:
def load_yamnet_model(artifact_path: Path):
    if artifact_path.name == 'saved_model.pb':
        model_root = artifact_path.parent
        saved_model = tf.saved_model.load(str(model_root))
        return saved_model
    return tf.keras.models.load_model(str(artifact_path))


yamnet_model = load_yamnet_model(yamnet_artifact)
type(yamnet_model)

In [ ]:
with CURATED_MANIFEST_PATH.open('r', encoding='utf-8') as handle:
    manifest = json.load(handle)

records = []
for dataset in manifest['datasets']:
    dataset_root = PROJECT_ROOT / 'external_data' / dataset['name']
    for clip in dataset['clips']:
        records.append({
            'dataset_name': dataset['name'],
            'audio_path': str(dataset_root.parent / clip['relative_path']),
            'class_slug': clip['class_slug'],
            'split': clip['split'],
            'clip_id': clip['clip_id'],
        })

manifest_df = pd.DataFrame(records)
manifest_df.head()

In [ ]:
def load_audio_file(audio_path: str) -> np.ndarray:
    waveform, sample_rate = sf.read(audio_path, dtype='float32', always_2d=False)
    if waveform.ndim > 1:
        waveform = np.mean(waveform, axis=1)
    if sample_rate != SAMPLE_RATE:
        waveform = librosa.resample(waveform, orig_sr=sample_rate, target_sr=SAMPLE_RATE)
    return waveform.astype(np.float32)


def resolve_embedding_tensor(output: object) -> np.ndarray:
    if isinstance(output, dict):
        for value in output.values():
            value_np = np.array(value)
            if value_np.ndim >= 2 and value_np.shape[-1] == 1024:
                return value_np
    if isinstance(output, (list, tuple)):
        for value in output:
            value_np = np.array(value)
            if value_np.ndim >= 2 and value_np.shape[-1] == 1024:
                return value_np
    value_np = np.array(output)
    if value_np.ndim >= 2 and value_np.shape[-1] == 1024:
        return value_np
    raise ValueError('Unable to resolve a 1024-d embedding tensor from the YAMNet output')


def extract_yamnet_embedding(waveform: np.ndarray) -> np.ndarray:
    input_tensor = tf.convert_to_tensor(waveform, dtype=tf.float32)
    if hasattr(yamnet_model, 'signatures') and yamnet_model.signatures:
        signature = next(iter(yamnet_model.signatures.values()))
        output = signature(input_tensor)
    else:
        output = yamnet_model(input_tensor)
    embedding_frames = resolve_embedding_tensor(output)
    if embedding_frames.ndim == 1:
        embedding_frames = np.expand_dims(embedding_frames, axis=0)
    return embedding_frames.mean(axis=0).astype(np.float32)

In [ ]:
embedding_rows = []
for row in manifest_df.to_dict(orient='records'):
    waveform = load_audio_file(row['audio_path'])
    embedding = extract_yamnet_embedding(waveform)
    embedding_rows.append({
        'clip_id': row['clip_id'],
        'class_slug': row['class_slug'],
        'split': row['split'],
        'embedding': embedding,
    })

embedding_df = pd.DataFrame(embedding_rows)
embedding_df.head()

In [ ]:
train_embeddings = embedding_df[embedding_df['split'] == 'train']
prototype_store = {}
for class_slug, class_frame in train_embeddings.groupby('class_slug'):
    prototype_store[class_slug] = np.stack(class_frame['embedding'].to_list(), axis=0).mean(axis=0).tolist()

with (YAMNET_OUTPUT_DIR / 'prototype_store.json').open('w', encoding='utf-8') as handle:
    json.dump(prototype_store, handle, indent=2)

print(f'Exported prototypes for {len(prototype_store)} classes')

In [ ]:
def prototype_predict(embedding: np.ndarray, prototypes: dict[str, list[float]]) -> tuple[str, float]:
    distances = {
        class_slug: float(np.sum((embedding - np.array(prototype, dtype=np.float32)) ** 2))
        for class_slug, prototype in prototypes.items()
    }
    predicted_label = min(distances, key=distances.get)
    return predicted_label, distances[predicted_label]


test_rows = embedding_df[embedding_df['split'] == 'test']
predictions = []
for row in test_rows.to_dict(orient='records'):
    predicted_label, distance = prototype_predict(row['embedding'], prototype_store)
    predictions.append({
        'clip_id': row['clip_id'],
        'true_label': row['class_slug'],
        'predicted_label': predicted_label,
        'distance': distance,
    })

predictions_df = pd.DataFrame(predictions)
print(classification_report(predictions_df['true_label'], predictions_df['predicted_label'], labels=CLASS_ORDER, zero_division=0))
predictions_df.head()

In [ ]:
artifact_summary = {
    'yamnet_kaggle_handle': YAMNET_KAGGLE_HANDLE,
    'yamnet_artifact': str(yamnet_artifact),
    'embedding_dim': PREPROCESSING_CONTRACT['embedding_contract']['yamnet_dim'],
    'class_order': CLASS_ORDER,
}

with (YAMNET_OUTPUT_DIR / 'artifact_summary.json').open('w', encoding='utf-8') as handle:
    json.dump(artifact_summary, handle, indent=2)

predictions_df.to_csv(YAMNET_OUTPUT_DIR / 'prototype_predictions.csv', index=False)
print(f'Wrote YAMNet outputs to {YAMNET_OUTPUT_DIR}')

In [ ]:
sample_row = manifest_df.iloc[0]
sample_waveform = load_audio_file(sample_row['audio_path'])
sample_embedding = extract_yamnet_embedding(sample_waveform)
sample_label, sample_distance = prototype_predict(sample_embedding, prototype_store)
print({'clip_id': sample_row['clip_id'], 'true_label': sample_row['class_slug'], 'predicted_label': sample_label, 'distance': sample_distance})